In [273]:
import pandas as pd
import json
import lib.data_handler as dh
import lib.display as dis
import lib.ollama_api as llm
import lib.utils as utils


# Downloading Datasets

In [274]:
# Players dataset from season 2017-18 to 2024-25
dh.download_dataset(
    out_dir="data/17-18_24-25/",
    path_kaggle="jacksonjohannessen/fifa-and-irl-soccer-player-data"
)

# Players dataset for season 2025-26
dh.download_dataset(
    out_dir="data/25-26/",
    path_kaggle="hubertsidorowicz/football-players-stats-2025-2026"
)


# Merging Datasets

In [275]:
data_players_2017_2025 = pd.read_csv("data/17-18_24-25/fifa_fbref_merged.csv")
data_players_2025_2026 = pd.read_csv("data/25-26/players_data-2025_2026.csv")
history_players = dh.concat(data_players_2017_2025, data_players_2025_2026)
history_players.to_csv("data/players_history.csv", index=False)
display(history_players.columns)

/var/folders/lx/1xhb8xws2gb4ss0_w168xmzh0000gn/T/ipykernel_86836/219913412.py:1: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  data_players_2017_2025 = pd.read_csv("data/17-18_24-25/fifa_fbref_merged.csv")


Index(['season', 'player', 'team', 'competition', 'nationality', 'position',
       'age', 'birth_year', 'appearances', 'starts', 'minutes', 'nineties',
       'goals_per90', 'assists_per90', 'goals_assists_per90',
       'non_penalty_goals_per90', 'non_penalty_goals_assists_per90',
       'penalty_attempts_per90', 'yellow_cards_per90', 'red_cards_per90',
       'shots_on_target_pct', 'shots_per90', 'shots_on_target_per90',
       'goals_per_shot', 'goals_per_shot_on_target', 'interceptions_per90',
       'tackles_won_per90', 'goals_against_per90',
       'shots_on_target_against_per90', 'saves_per90', 'save_pct',
       'wins_per90', 'draws_per90', 'losses_per90', 'clean_sheets_per90',
       'clean_sheet_pct', 'keeper_penalty_attempts_per90',
       'penalties_allowed_per90', 'penalties_saved_per90',
       'penalties_missed_per90'],
      dtype='object')

# Dataset Fantacalcio

In [276]:
fanta_players = pd.read_excel("Listone_Fantacalcio_Stagione_2026_27.xlsx", header=1)
fanta_players.to_csv("data/Listone_Fantacalcio_Stagione_2026_27.csv", index=False)
fanta_players = pd.read_csv("data/Listone_Fantacalcio_Stagione_2026_27.csv")
fanta_players.shape

(509, 13)

In [277]:
fanta_players.columns

Index(['Id', 'R', 'RM', 'Nome', 'Squadra', 'Qt.A', 'Qt.I', 'Diff.', 'Qt.A M',
       'Qt.I M', 'Diff.M', 'FVM', 'FVM M'],
      dtype='object')

In [278]:
fanta_players.head(10)

,Id,R,RM,Nome,Squadra,Qt.A,Qt.I,Diff.,Qt.A M,Qt.I M,Diff.M,FVM,FVM M
0,5841,P,Por,Svilar,Roma,18,18,0,18,18,0,65,65
1,5116,P,Por,Martinez Jo.,Inter,17,17,0,17,17,0,63,63
2,4431,P,Por,Carnesecchi,Atalanta,16,16,0,16,16,0,52,52
3,6966,P,Por,Butez,Como,16,16,0,16,16,0,56,56
4,4964,P,Por,Vicario,Juventus,16,16,0,16,16,0,55,55
5,4312,P,Por,Maignan,Milan,15,15,0,15,15,0,50,50
6,2521,P,Por,De Gea,Fiorentina,13,13,0,13,13,0,45,45
7,572,P,Por,Meret,Napoli,11,11,0,11,11,0,45,45
8,133,P,Por,Skorupski,Bologna,10,10,0,10,10,0,37,37
9,4360,P,Por,Caprile,Cagliari,9,9,0,9,9,0,27,27


In [279]:
fanta_players["RM"].unique()

array(['Por', 'E;W', 'E', 'Dc', 'Dd;Dc', 'Dd;E', 'Ds;Dc', 'Dd;Ds;E',
       'Ds;E', 'B;Dd;E', 'B;Ds;E', 'Dd;Ds;Dc', 'T;A', 'C;T', 'M;C', 'W;A',
       'T', 'C', 'W;T', 'W', 'E;C', 'C;W', 'Pc', 'A'], dtype=object)

# Normalization phase

In [280]:
# Casting
history_players["age"] = pd.to_numeric(history_players["age"], errors="coerce").astype("Int64")
history_players["birth_year"] = pd.to_numeric(history_players["birth_year"], errors="coerce").astype("Int64")
history_players["minutes"] = pd.to_numeric(history_players["minutes"], errors="coerce").astype("Int64")

dis.print_unique_values(history_players, max_values=10)


Dataset shape: 31,275 rows × 40 columns

[1/40] season
--------------------------------------------------------------------------------
Type: object | Unique: 9 | Missing: 0
  • '2023-24'
  • '2022-23'
  • '2021-22'
  • '2020-21'
  • '2019-20'
  • '2018-19'
  • '2017-18'
  • '2024-25'
  • '2025-26'

[2/40] player
--------------------------------------------------------------------------------
Type: object | Unique: 10,728 | Missing: 0
  • 'Kylian Mbappe'
  • 'Erling Haaland'
  • 'Kevin De Bruyne'
  • 'Lionel Messi'
  • 'Robert Lewandowski'
  • 'Thibaut Courtois'
  • 'Harry Kane'
  • 'Vinicius Junior'
  • 'Alisson'
  • 'Rodri'
  ... 10,718 additional values not displayed

[3/40] team
--------------------------------------------------------------------------------
Type: object | Unique: 301 | Missing: 0
  • 'Paris S-G'
  • 'Manchester City'
  • 'Inter Miami'
  • 'Barcelona'
  • 'Real Madrid'
  • 'Bayern Munich'
  • 'Liverpool'
  • 'Manchester Utd'
  • 'Napoli'
  • 'Atalanta'
  ... 291 a

In [281]:
# Normalization of the competition names
history_players["competition"] = history_players["competition"].apply(
    lambda x: " ".join(x.split(" ")[1:]) if isinstance(x, str) and "-" not in x and len(x.split(" ")) > 1 else x
)
history_players["competition"] = history_players["competition"].apply(
    lambda x: " ".join(x.split("-")[1:]) if isinstance(x, str) and "-" in x and len(x.split("-")) > 1 else x
)
history_players["competition"].unique()

array(['Ligue 1', 'Premier League', 'MLS', 'La Liga', 'Bundesliga',
       'Serie A', 'Super Lig', 'Primeira Liga', 'Jupiler Pro League',
       'Eredivisie'], dtype=object)

In [282]:
# Normalization of the nationalities
history_players["nationality"] = history_players["nationality"].apply(
    lambda x: x.split(" ")[1] if isinstance(x, str) and len(x.split(" ")) > 1 else x
)
history_players["nationality"].unique()

array(['FRA', 'NOR', 'BEL', 'ARG', 'POL', 'ENG', 'BRA', 'ESP', 'POR',
       'GER', 'NED', 'EGY', 'URU', 'NGA', 'SVN', 'ITA', 'SUI', 'CRO',
       'KOR', 'GEO', 'SCO', 'CMR', 'CRC', 'AUT', 'TUR', 'GHA', 'ALG',
       'MAR', 'DEN', 'COL', 'HUN', 'SVK', 'CAN', 'SRB', 'CZE', 'BIH',
       'CIV', 'UKR', 'SEN', 'MNE', 'IRN', 'SWE', 'BFA', 'MEX', 'GNB',
       'TUN', 'JPN', 'LBY', 'CHI', 'FIN', 'ARM', 'CTA', 'PAR', 'MOZ',
       'KVX', 'ECU', 'COD', 'GAB', 'MLI', 'USA', 'IDN', 'JAM', 'MKD',
       'RUS', 'MLT', 'GRE', 'PER', 'ANG', 'AUS', 'SUR', 'ALB', 'ISR',
       'ZIM', 'WAL', 'VEN', 'GUI', 'TOG', 'IRL', 'LUX', 'COM', 'ROU',
       'EQG', 'BDI', 'HAI', 'DOM', 'CPV', 'NIR', 'NZL', 'CUW', 'GLP',
       'ISL', 'KEN', 'HON', 'GAM', 'PLE', 'BEN', 'MTQ', 'MAD', 'CGO',
       'GUF', 'PAN', 'UZB', 'ZAM', 'BUL', 'CYP', 'MDA', 'GRN', 'MAS',
       'AZE', 'SLE', 'PHI', 'RSA', nan, 'IRQ', 'GUA', 'MTN', 'SLV', 'LTU',
       'SKN', 'PUR', 'CUB', 'BOE', 'TRI', 'TAN', 'SYR', 'KAZ', 'UGA',
       'LVA', '

In [283]:
# Check of equal teams names
history_teams = history_players[history_players["competition"] == "Serie A"]["team"].unique()
for team in fanta_players["Squadra"].unique().tolist():
    if team not in history_teams:
        raise Exception(f"Team {team} absent in:\n{history_teams}")

# Players matching
For our pourposes are needed the stats of only the current players. So we have to match the current season players names with the ones in the previous seasons.

In [284]:
filtered_history, unmatched_players = dh.filter_history_exact_matches(
    history_df=history_players,
    fanta_df=fanta_players,
    history_name_col="player",
    fanta_name_col="Nome",
)
unmatched_players

,Id,R,RM,Nome,Squadra,Qt.A,Qt.I,Diff.,Qt.A M,Qt.I M,Diff.M,FVM,FVM M
0,5841,P,Por,Svilar,Roma,18,18,0,18,18,0,65,65
1,5116,P,Por,Martinez Jo.,Inter,17,17,0,17,17,0,63,63
2,4431,P,Por,Carnesecchi,Atalanta,16,16,0,16,16,0,52,52
3,6966,P,Por,Butez,Como,16,16,0,16,16,0,56,56
4,4964,P,Por,Vicario,Juventus,16,16,0,16,16,0,55,55
...,...,...,...,...,...,...,...,...,...,...,...,...,...
504,7161,A,Pc,Bayo V.,Udinese,1,1,0,1,1,0,3,3
505,7408,A,W;A,Lisman,Venezia,1,1,0,1,1,0,1,1
506,7482,A,Pc,Lauberbach,Venezia,1,1,0,1,1,0,1,1
507,7561,A,A,Lontani,Parma,1,1,0,1,1,0,3,3


In [285]:
history_players_modified, fanta_players_modified = dh.filter_history_relaxed_matches(history_players, fanta_players,)

# The relaxed matcher can duplicate an original row when it finds multiple candidates
matched_mask = fanta_players_modified["normalized_name"].isin(history_players_modified["normalized_name"].dropna())
matched_indices = fanta_players_modified.index[matched_mask].unique()
matched_history_names = set(fanta_players_modified.loc[matched_mask, "normalized_name"])

# Keep only historical rows associated with at least one Fantacalcio player.
filtered_history = history_players_modified.loc[
    history_players_modified["normalized_name"].isin(matched_history_names)
].copy()

# Count matched and unmatched players using the original Fantacalcio rows.
matched_fanta = fanta_players.loc[
    fanta_players.index.isin(matched_indices)
].copy()
unmatched_fanta = fanta_players.loc[
    ~fanta_players.index.isin(matched_indices)
].copy()
unmatched_fanta["normalized_name"] = unmatched_fanta["Nome"].apply(dh.normalize_player_name)

# Print results
print("Unmatched players:")
print(unmatched_fanta[["Nome", "normalized_name"]].to_string(index=False))
print()
print(f"Matched: {matched_fanta.shape[0]}/{fanta_players.shape[0]}")
print(f"Unmatched: {unmatched_fanta.shape[0]}/{fanta_players.shape[0]}")

Unmatched players:
              Nome    normalized_name
           Daffara            daffara
         Palmisani          palmisani
       Desplanches        desplanches
          Happonen           happonen
         Tornqvist          tornqvist
             Lolic              lolic
             Stolz              stolz
         Pinsoglio          pinsoglio
          Renzetti           renzetti
          Torriani           torriani
       Pizzignacco        pizzignacco
          Strajnar           strajnar
          De Marzi           de marzi
          Mascardi           mascardi
           Siviero            siviero
             Piana              piana
            Grandi             grandi
             Pozzi              pozzi
           Vismara            vismara
          Satalino           satalino
             Penev              penev
           Pisseri            pisseri
             Bleve              bleve
             Kaiki              kaiki
             Viery             

In [286]:
# Rename the Fantacalcio fields already standardized in the historical dataset
fanta_column_renames = {
    "Id": "id",
    "Nome": "fanta_player",
    "R": "fanta_role",
    "RM": "mantra_role",
}
fanta_columns = ["normalized_name"] + fanta_players.columns.tolist()

# Build the mapping with every field from the Fantacalcio dataset
fanta_mapping = (
    fanta_players_modified.loc[
        fanta_players_modified["normalized_name"].isin(matched_history_names),
        fanta_columns,
    ]
    .drop_duplicates()
    .rename(columns=fanta_column_renames)
)

# Remove the historical ID before assigning the Fantacalcio ID.
history_without_id = history_players_modified.drop(columns=["id"], errors="ignore")

# Keep matched historical players and attach Fantacalcio information.
filtered_history_players = history_without_id.merge(fanta_mapping, on="normalized_name", how="inner")

# Replace the historical name with the Fantacalcio name.
filtered_history_players["player"] = filtered_history_players.pop("fanta_player")

# Find Fantacalcio players without historical matches.
matched_fanta_ids = set(fanta_mapping["id"].dropna())
unmatched_fanta = fanta_players.loc[~fanta_players["Id"].isin(matched_fanta_ids)].copy()

# Prepare every Fantacalcio field for the current-season rows
current_fanta_data = fanta_players.rename(columns=fanta_column_renames).copy()
current_fanta_data["player"] = current_fanta_data.pop("fanta_player")

# Create empty historical rows for every current Fantacalcio player
current_fanta_history = pd.DataFrame(
    pd.NA,
    index=range(len(fanta_players)),
    columns=filtered_history_players.columns,
)

# Add every Fantacalcio field and leave the historical statistics empty
for column in current_fanta_data.columns:
    current_fanta_history[column] = current_fanta_data[column].to_numpy()

current_fanta_history["team"] = current_fanta_history["Squadra"]
current_fanta_history["competition"] = "Serie A"
current_fanta_history["season"] = "2026-27"
current_fanta_history["normalized_name"] = current_fanta_history["player"].apply(dh.normalize_player_name)

# Append the complete current Fantacalcio list to the historical dataset
filtered_history_players = pd.concat(
    [filtered_history_players, current_fanta_history],
    ignore_index=True,
)

print(f"Matched historical rows: {len(filtered_history_players) - len(current_fanta_history)}")
print(f"Current Fantacalcio players added: {len(current_fanta_history)}")
print(f"Unmatched Fantacalcio players: {len(unmatched_fanta)}")
print(f"Final rows: {len(filtered_history_players)}")

filtered_history_players.head()

Matched historical rows: 4097
Current Fantacalcio players added: 509
Unmatched Fantacalcio players: 63
Final rows: 4606


,season,player,team,competition,nationality,position,age,birth_year,appearances,starts,...,mantra_role,Squadra,Qt.A,Qt.I,Diff.,Qt.A M,Qt.I M,Diff.M,FVM,FVM M
0,2023-24,De Bruyne,Manchester City,Premier League,BEL,MF,32,1991,18,15,...,T,Napoli,15,15,0,14,14,0,98,95
1,2022-23,De Bruyne,Manchester City,Premier League,BEL,MF,31,1991,32,28,...,T,Napoli,15,15,0,14,14,0,98,95
2,2021-22,De Bruyne,Manchester City,Premier League,BEL,MF,30,1991,30,25,...,T,Napoli,15,15,0,14,14,0,98,95
3,2020-21,De Bruyne,Manchester City,Premier League,BEL,MF,29,1991,25,23,...,T,Napoli,15,15,0,14,14,0,98,95
4,2019-20,De Bruyne,Manchester City,Premier League,BEL,MF,28,1991,35,32,...,T,Napoli,15,15,0,14,14,0,98,95


In [287]:
# Ordering of the columns
ordered_columns = filtered_history_players.columns.tolist()

# Move "id" to the first position.
ordered_columns.remove("id")
ordered_columns.insert(0, "id")

# Move "fanta_role" immediately after "position".
ordered_columns.remove("fanta_role")
ordered_columns.insert(ordered_columns.index("position")+1, "fanta_role")

# Move "mantra_role" immediately after "fanta_role"
ordered_columns.remove("mantra_role")
ordered_columns.insert(ordered_columns.index("fanta_role")+1, "mantra_role")

filtered_history_players = filtered_history_players[ordered_columns]
filtered_history_players.head(10)

,id,season,player,team,competition,nationality,position,fanta_role,mantra_role,age,...,normalized_name,Squadra,Qt.A,Qt.I,Diff.,Qt.A M,Qt.I M,Diff.M,FVM,FVM M
0,2517,2023-24,De Bruyne,Manchester City,Premier League,BEL,MF,C,T,32,...,kevin de bruyne,Napoli,15,15,0,14,14,0,98,95
1,2517,2022-23,De Bruyne,Manchester City,Premier League,BEL,MF,C,T,31,...,kevin de bruyne,Napoli,15,15,0,14,14,0,98,95
2,2517,2021-22,De Bruyne,Manchester City,Premier League,BEL,MF,C,T,30,...,kevin de bruyne,Napoli,15,15,0,14,14,0,98,95
3,2517,2020-21,De Bruyne,Manchester City,Premier League,BEL,MF,C,T,29,...,kevin de bruyne,Napoli,15,15,0,14,14,0,98,95
4,2517,2019-20,De Bruyne,Manchester City,Premier League,BEL,MF,C,T,28,...,kevin de bruyne,Napoli,15,15,0,14,14,0,98,95
5,2517,2018-19,De Bruyne,Manchester City,Premier League,BEL,MF,C,T,27,...,kevin de bruyne,Napoli,15,15,0,14,14,0,98,95
6,2517,2017-18,De Bruyne,Manchester City,Premier League,BEL,MF,C,T,26,...,kevin de bruyne,Napoli,15,15,0,14,14,0,98,95
7,2517,2024-25,De Bruyne,Manchester City,Premier League,BEL,"MF,FW",C,T,33,...,kevin de bruyne,Napoli,15,15,0,14,14,0,98,95
8,2517,2025-26,De Bruyne,Napoli,Serie A,BEL,MF,C,T,34,...,kevin de bruyne,Napoli,15,15,0,14,14,0,98,95
9,5792,2023-24,Ederson D.S.,Atalanta,Serie A,BRA,MF,C,M;C,24,...,ederson,Atalanta,12,12,0,13,13,0,45,48


In [288]:
# Store data
unmatched_fanta.to_csv("data/matched_unmatched_datasets/unmatched_fanta_players.csv", index=False)
matched_fanta.to_csv("data/matched_unmatched_datasets/matched_fanta_players.csv", index=False)

filtered_history_players.to_csv("data/filtered_history_players.csv", index=False)

In [289]:
prompt = """
Task:
Match one player from the 2026-27 Fantacalcio dataset with candidate historical
records covering seasons from 2017-18 to 2025-26.

Matching rules:
1. Determine whether each candidate represents the same real football player
   as the Fantacalcio player.
2. The current team does not need to match the historical team because players
   can transfer between teams and competitions.
3. A matching current team is useful evidence, but a different team must never
   be used by itself to reject a candidate.
4. Give primary importance to:
   - surname and name compatibility;
   - first-name initial, when available;
   - birth year and plausible age progression;
   - nationality;
   - compatible playing position.
5. Position labels from the two datasets may differ and must be treated as
   compatible when they describe similar roles.
6. Nationality representations such as "NGA" and "ng NGA" are equivalent.
7. Fantacalcio names are frequently abbreviated:
   - surname only: "Basic" can match "Toma Basic";
   - surname plus first-name initial: "Perez K." can match "Kike Perez";
   - accents may be omitted: "Kone" can match "Koné";
   - name and surname order may differ.
8. Do not reject a candidate only because the Fantacalcio name omits the first
   name or replaces it with an initial.
9. The initial must be compatible when both the candidate's first name and the
   Fantacalcio initial are available.
10. Multiple candidate rows from different seasons may represent the same
    player. Return all matching candidate IDs.
11. Evaluate every candidate independently. Do not stop after finding the first
    match.
12. Return an empty list only if none of the candidates plausibly represents
    the same person.
13. Never invent candidate IDs. Return only IDs appearing in the input.
14. Do not infer identity or nationality solely from how a surname sounds.
15. If the role is the same and the name of the fanta_player is abbreviated 
    and the surname of the fanta_player is in the name of the candidate,
    try to infer the player's name based on the team or the competition and retry with the full name.

Fantacalcio Mantra roles:
- Por = goalkeeper
- Dc, Dd, Ds, B = defender
- E, M, C = midfielder
- W, T, A, Pc = attacking player
- Roles separated by ";" indicate multiple compatible roles.
- Pc means centre-forward and never goalkeeper.

Example:
A Fantacalcio player named "De Gea" playing for Fiorentina can match historical
records named "David de Gea" playing for Manchester United. The different team
is explained by a transfer and does not invalidate the identity match.

Input JSON:
{request_data}

Return only one valid JSON object with exactly these fields:
{
    "matched_candidate_ids": [integer candidate IDs],
    "confidence": number between 0 and 1,
    "reason": "short explanation"
}

If no candidate matches, return:
{
    "matched_candidate_ids": [],
    "confidence": 0.0,
    "reason": "short explanation"
}
"""
'''
failed_matches: pd.DataFrame = pd.read_csv("data/failed_matches.csv")
failed_fanta_players = (
    players[
        players["Nome"].isin(failed_matches["Nome"].dropna().unique())
    ].copy().reset_index(drop=True)
)
filtered_history, refailed_matches = llm.get_filtered_history_with_llm_matches(failed_fanta_players, history_players, prompt)'''

'\nfailed_matches: pd.DataFrame = pd.read_csv("data/failed_matches.csv")\nfailed_fanta_players = (\n    players[\n        players["Nome"].isin(failed_matches["Nome"].dropna().unique())\n    ].copy().reset_index(drop=True)\n)\nfiltered_history, refailed_matches = llm.get_filtered_history_with_llm_matches(failed_fanta_players, history_players, prompt)'

In [290]:
# Create a DataFrame for the failed matches
'''refailed_matches_df = pd.DataFrame(refailed_matches)
refailed_matches_df.to_csv("data/refailed_matches.csv", index=False)

for _, fanta_row in refailed_matches_df.iterrows():
    if not str(fanta_row["reason"]).startswith("No candidates match"):
        print(fanta_row["fanta_player"])
        print(fanta_row["reason"])
refailed_matches_df.shape'''

'refailed_matches_df = pd.DataFrame(refailed_matches)\nrefailed_matches_df.to_csv("data/refailed_matches.csv", index=False)\n\nfor _, fanta_row in refailed_matches_df.iterrows():\n    if not str(fanta_row["reason"]).startswith("No candidates match"):\n        print(fanta_row["fanta_player"])\n        print(fanta_row["reason"])\nrefailed_matches_df.shape'